In [8]:

import sys
import os
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Default dataset path
CSV_PATH = "train.csv"


def train(csv_path):
    # 1. Load training dataset
    if not os.path.exists(csv_path):
        print("Error: Dataset not found:", csv_path)
        return

    data = pd.read_csv(csv_path)

    # 2. Check target column
    if "Attrition" not in data.columns:
        print("Error: 'Attrition' column not found.")
        print("Available columns:", list(data.columns))
        return

    print("Dataset loaded successfully!")
    print("Dataset shape:", data.shape)

    # 3. Separate input and target
    X = data.drop("Attrition", axis=1)
    y = data["Attrition"]

    # 4. Identify numerical and categorical columns
    num_cols = X.select_dtypes(include="number").columns
    cat_cols = X.select_dtypes(exclude="number").columns

    # 5. Preprocess data
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, num_cols),
        ("categorical", categorical_pipeline, cat_cols)
    ])

    # 6. Create Random Forest model
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

    # 7. Split data for validation
    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # 8. Train and validate
    print("\nTraining model...")
    model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    print("\nValidation Accuracy:",
          round(accuracy_score(y_val, predictions) * 100, 2), "%")
    print("\nClassification Report:")
    print(classification_report(y_val, predictions, zero_division=0))

    # 9. Retrain on the complete dataset
    print("Training final model on complete dataset...")
    model.fit(X, y)

    # 10. Save trained model
    joblib.dump(model, "random_forest_attrition_model.joblib")
    print("Model saved as random_forest_attrition_model.joblib")

    # 11. Load test dataset if available
    test_path = os.path.join(os.path.dirname(csv_path), "test.csv")

    if os.path.exists(test_path):
        test_data = pd.read_csv(test_path)

        # Separate actual labels if available
        if "Attrition" in test_data.columns:
            y_test = test_data["Attrition"]
            X_test = test_data.drop("Attrition", axis=1)
        else:
            y_test = None
            X_test = test_data.copy()

        # Match the training columns
        X_test = X_test.reindex(columns=X.columns)

        # Generate predictions
        test_predictions = model.predict(X_test)

        results = pd.DataFrame({
            "Predicted_Attrition": test_predictions
        })

        if y_test is not None:
            results.insert(0, "Actual_Attrition", y_test.values)

            print("\nTest Accuracy:",
                  round(accuracy_score(y_test, test_predictions) * 100, 2), "%")
            print(classification_report(
                y_test, test_predictions, zero_division=0
            ))

        results.to_csv("test_predictions.csv", index=False)
        print("Predictions saved as test_predictions.csv")

    else:
        print("test.csv not found. Skipping test predictions.")

    print("\nTraining completed successfully!")


# 12. Main program
if __name__ == "__main__":
    args = [
        a for a in sys.argv[1:]
        if a.lower().endswith(".csv")
    ]

    train(args[0] if args else CSV_PATH)


Dataset loaded successfully!
Dataset shape: (59598, 24)

Training model...

Validation Accuracy: 74.44 %

Classification Report:
              precision    recall  f1-score   support

        Left       0.73      0.74      0.73      5668
      Stayed       0.76      0.75      0.75      6252

    accuracy                           0.74     11920
   macro avg       0.74      0.74      0.74     11920
weighted avg       0.74      0.74      0.74     11920

Training final model on complete dataset...
Model saved as random_forest_attrition_model.joblib

Test Accuracy: 74.58 %
              precision    recall  f1-score   support

        Left       0.73      0.74      0.73      7032
      Stayed       0.76      0.76      0.76      7868

    accuracy                           0.75     14900
   macro avg       0.75      0.75      0.75     14900
weighted avg       0.75      0.75      0.75     14900

Predictions saved as test_predictions.csv

Training completed successfully!
